# Experiment 1 - per-configuration diagnostic figures

Writes one folder per config under `experiment_figs/`:
- `w_comparison_cell_*.png` - estimated vs model w (Hovmoller + time series), built from saved arrays (fast, no model needed).
- `velocity_map.png` - depth/time-mean U, V, W with the array overlaid (loads the model once).

Run in place after `run_experiment.py`.

In [ ]:
import os, sys, json, glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

REPO = "/home/edavenport/analysis/tpose24-osse"
HERE = os.path.join(REPO, "experiment_1")
sys.path.insert(0, REPO)
import osse_tools as ot

DATA = os.path.join(HERE, "data")
EFIG = os.path.join(HERE, "experiment_figs"); os.makedirs(EFIG, exist_ok=True)
plt.rcParams["figure.dpi"] = 110
m = pd.read_csv(os.path.join(DATA, "metrics.csv"))
print(f"{len(m)} cells / {m.config.nunique()} configs")

## Per-cell w comparison (from saved arrays)

In [ ]:
# Per-cell w comparison (Hovmoller + time series) from the saved arrays - no model load.
for _, r in m.iterrows():
    ds = xr.open_dataset(os.path.join(HERE, r.nc_path))
    fig = ot.plot_w_comparison(ds.w_est, ds.w_model, point_depth=-50)
    fig.suptitle(f"{r.config}  |  cell {r.center_lat:+.1f}N  |  "
                 f"RMS/sigma={r.norm_rms:.2f}  r={r['corr']:.2f}", y=1.01, fontsize=12)
    outdir = os.path.join(EFIG, r.config); os.makedirs(outdir, exist_ok=True)
    fig.savefig(os.path.join(outdir, f"w_comparison_cell_{r.center_lat:+.2f}.png"),
                dpi=130, bbox_inches="tight")
    plt.close(fig)
print("wrote per-cell w_comparison figures")

## Per-config velocity maps (loads the model)

In [ ]:
# Per-config velocity/vorticity context maps. These need the model, so load it once.
RUN_DIR = "/data/SO3/edavenport/tpose24/oct2012_3month_transp_cons"
ITERS   = list(range(36, 26173, 36))
ds_model = ot.load_model(RUN_DIR, ITERS).sel(time=slice("2012-10-11", None))

cfg_paths = sorted(glob.glob(os.path.join(HERE, "configs", "**", "*.json"), recursive=True))
for path in cfg_paths:
    cfg = json.load(open(path))
    cells = ot.load_cells(path)
    positions = sorted({p for _, pos in cells for p in pos})
    cells_plot = [(f"{cl:+.1f}", pos, f"C{i}") for i, (cl, pos) in enumerate(cells)]
    fig = ot.plot_velocity_map(ds_model, positions, max_depth=70, cells=cells_plot)
    fig.suptitle(f"{cfg['name']}  ({cfg['description']})", fontsize=11, y=1.02)
    outdir = os.path.join(EFIG, cfg["name"]); os.makedirs(outdir, exist_ok=True)
    fig.savefig(os.path.join(outdir, "velocity_map.png"), dpi=130, bbox_inches="tight")
    plt.close(fig)
print("wrote per-config velocity maps")